In [ ]:
%load_ext autoreload
%autoreload 2

# Load dependencies

In [ ]:
# if src modules imported
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

In [ ]:
import torch
import matplotlib as mpl
from operator import itemgetter
from torch import nn,optim
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from datasets import load_dataset


# src modules
from src.datasets import show_image, inplace, show_images
from src.conv import to_device, def_device, conv
from src.datasets import collate_dict
from src.training import fit

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
torch.manual_seed(1)
mpl.rcParams['image.cmap'] = 'gray'
mpl.rcParams['figure.dpi'] = 70

# Load and Transform data

In [ ]:
x,y = 'image','label'
name = "fashion_mnist"
dsd = load_dataset(name)

In [ ]:
@inplace
def transformi(b): b[x] = [TF.to_tensor(o) for o in b[x]]
# TODO: what are other ways of doing this without using `inplace`?
# for example, by setting `columns` and `output_all_columns` of `with_transform` to proper values
tds = dsd.with_transform(transformi)
ds = tds['train']
img = ds[0]['image']
show_image(img, figsize=(1,1));

# Create data loaders

In [ ]:
bs = 256
cf = collate_dict(ds)
def data_loaders(dsd, bs, **kwargs):
    return {k:DataLoader(v, bs, **kwargs, num_workers=8) for k,v in dsd.items()}
dls = data_loaders(tds, bs, collate_fn=lambda batch: to_device(cf(batch)))
dt = dls['train']
dv = dls['test']

xb,yb = next(iter(dt))
print(f"first batch shape: {xb.shape} | {yb.shape}")
print("======================")
labels = ds.features[y].names
print(labels)
print("======================")
preveiw_sample_size = 16
lbl_getter = itemgetter(*yb[:preveiw_sample_size])
titles = lbl_getter(labels)
show_images(xb[:preveiw_sample_size], imsize=1.7, titles=titles);

# Build a classifier network

In [ ]:
cnn = nn.Sequential(
    conv(1 ,4),            #14x14
    conv(4 ,8),            #7x7
    conv(8 ,16),           #4x4
    conv(16,16),           #2x2
    conv(16,10, act=False),
    nn.Flatten()).to(def_device)
opt = optim.SGD(cnn.parameters(), lr = 0.4)
fit(cnn,  dt, dv, opt, F.cross_entropy, 5)

- Why is This too slow?
    - in each batch the dataloader must read and decode image png files and stack them together
    - one option is increase the `num_workers` in the dataloader creation method:
    ```
    def data_loaders(dsd, bs, **kwargs):
    return {k:DataLoader(v, bs, **kwargs, num_workers=4) for k,v in dsd.items()}
    ```
    - another option is to separate
        - **loading/decoding** images (before and outside `dataloader`)
        - **stacking** (inside `dataloader`) --> can be done in multi processes
        - and **device mapping** (`to_device`) inside the train loop
            - to avoid issues that might raise beacuse of mapping data to gpu from multiple processes

# Autoencoder

In [ ]:
def deconv(ni, nf, ks=3, act=True):
    layers = [nn.UpsamplingNearest2d(scale_factor=2),
              nn.Conv2d(ni, nf, stride=1, kernel_size=ks, padding=ks//2)]
    if act: layers.append(nn.ReLU())
    return nn.Sequential(*layers)

ae = nn.Sequential(   #28x28
    nn.ZeroPad2d(2),  #32x32
    conv(1,2),        #16x16
    conv(2,4),        #8x8
#     conv(4,8),        #4x4
#     deconv(8,4),      #8x8
    deconv(4,2),      #16x16
    deconv(2,1, act=False), #32x32
    nn.ZeroPad2d(-2), #28x28
    nn.Sigmoid()
).to(def_device)

opt = optim.SGD(ae.parameters(), lr=0.01)
fit(ae, dt, dv,opt,F.mse_loss,5, ae_loss=True)
opt = optim.SGD(ae.parameters(), lr=0.1)
fit(ae, dt, dv,opt,F.mse_loss,5, ae_loss=True)

In [ ]:
p = ae(xb)
show_images(p[:16].data.cpu(), imsize=1.5)

In [ ]:
show_images(xb[:16].data.cpu(), imsize=1.5)